# PantheraLAB 🐆 — Prueba mejorada de detección real (v2)

**Objetivo:** detectar tu perro u otro animal usando MegaDetector V6 y, **solo si el recuadro es útil**, sugerir especie con SpeciesNet.

**Primera prueba:** el modelo compacto marcó solo una parte pequeña del rostro (52,49 % de confianza) y SpeciesNet clasificó ese pequeño recorte como `blank` (98,57 %). **Eso NO significa que la foto esté vacía** y no demuestra identificación del perro.

Esta revisión usa MegaDetector V6 Extra, comprueba la calidad de las cajas, corrige la descarga de SpeciesNet en Colab y evita presentar `blank` como especie identificada.

1. Elegí **Entorno de ejecución → Ejecutar todas**.
2. Subí una foto tuya del animal (idealmente de cuerpo completo, con el animal visible).
3. Mirá la imagen con los recuadros, el diagnóstico y el informe JSON.

Usar una GPU de Colab es opcional (si está disponible). **No usa PantheraLIVE ni instala modelos en Render.** No subas imágenes de personas ni datos sensibles. La fotografía se procesa en la sesión temporal de Colab.

Modelos y documentación: https://github.com/microsoft/MegaDetector · https://github.com/google/cameratrapai

## 1. Instalar MegaDetector y SpeciesNet
Solo en la sesión de Colab. Puede tardar varios minutos en descargar paquetes.

In [ ]:
%pip -q install PytorchWildlife speciesnet Pillow

## 2. Elegir la foto del perro u otro animal
Se ejecuta una única prueba sobre tu fotografía. Recomendación: JPG, PNG o WEBP de hasta 12 MB.

In [ ]:
from google.colab import files
from IPython.display import display
from PIL import Image, ImageOps
from pathlib import Path
from io import BytesIO
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No se seleccionó una fotografía.")
name, payload = next(iter(uploaded.items()))
if len(payload)>12*1024*1024:
    raise ValueError("Fotografía mayor que 12 MB.")
photo = ImageOps.exif_transpose(Image.open(BytesIO(payload))).convert("RGB")
if photo.width*photo.height>25_000_000:
    raise ValueError("Fotografía mayor que 25 megapíxeles.")
folder = Path("/content/pantheralab_prueba_real")
folder.mkdir(parents=True,exist_ok=True)
file_path = folder/"fotograma.jpg"
photo.save(file_path,quality=92)
preview = photo.copy()
preview.thumbnail((820,820))
display(preview)
print("Fotografía preparada:",photo.width,"x",photo.height)

## 3. MegaDetector V6 — variante de mayor capacidad

Probamos `MDV6-yolov10-e` (29,5 M de parámetros, más preciso en las pruebas publicadas que la variante compacta anterior). Aumenta el consumo de memoria y el tiempo de inferencia. Usamos un umbral de **0,20** para reducir omisiones a costa de más falsos positivos.

**Revisión esencial:** una caja sobre solo un ojo, la nariz o una región diminuta **no equivale a localizar todo el perro**. El diagnóstico de tamaño de recuadro es una heurística de laboratorio, no un umbral universal para fauna.

In [ ]:
import torch
from PIL import ImageDraw
from PytorchWildlife.models import detection as pw_detection

device = "cuda" if torch.cuda.is_available() else "cpu"
variant = "MDV6-yolov10-e"
threshold = 0.20
print("Modelo:", variant, "| Procesador:", device)
print("Descargando pesos si hace falta. La variante extra puede tardar más.")
model = pw_detection.MegaDetectorV6(
    device=device, pretrained=True, version=variant
)
prediction = model.single_image_detection(str(file_path),det_conf_thres=threshold)
objects = prediction.get("detections")
names = {0:"animal",1:"person",2:"vehicle"}
detections = []
if objects is not None:
    for xyxy,conf,class_id in zip(objects.xyxy,objects.confidence,objects.class_id):
        box = [int(round(float(v))) for v in xyxy]
        xmin,ymin,xmax,ymax=box
        area = max(0,xmax-xmin)*max(0,ymax-ymin)
        ratio = area / (photo.width*photo.height)
        detections.append({
            "class_name":names.get(int(class_id),"unknown"),
            "confidence":round(float(conf),4),
            "bbox_xyxy":box,
            "box_fraction_of_image":round(ratio,5)
        })

annotated=photo.copy()
draw=ImageDraw.Draw(annotated)
for detection in detections:
    box=detection["bbox_xyxy"]
    draw.rectangle(box,outline="#00db9d",width=max(3,int(min(photo.size)/200)))
    draw.text((box[0]+3,max(0,box[1]-16)),
              detection["class_name"].upper()+" "+str(round(detection["confidence"]*100))+"%",
              fill="#00db9d")
view=annotated.copy()
view.thumbnail((920,920))
display(view)
annotated.save(folder/"detecciones.jpg",quality=92)
print("Animales:",sum(d["class_name"]=="animal" for d in detections))
print("Personas:",sum(d["class_name"]=="person" for d in detections))
print("Vehículos:",sum(d["class_name"]=="vehicle" for d in detections))
for d in detections:
    print(d["class_name"].upper(),"confianza:",round(d["confidence"]*100,1),"%",
          "| tamaño de caja:",round(d["box_fraction_of_image"]*100,2),"% de la foto")
    if d["class_name"]=="animal" and d["box_fraction_of_image"]<0.01:
        print("⚠ Caja muy pequeña en esta prueba; la identificación de especie puede fallar.")

## 4. SpeciesNet — carga corregida para Google Colab

El error anterior «Read-only file system» ocurría porque SpeciesNet intentaba descargar **otro MegaDetector** dentro de la carpeta de KaggleHub, que en Colab puede ser de solo lectura.

**Corrección:** preparamos una carpeta temporal con permisos de escritura para los metadatos y reutilizamos los pesos oficiales del clasificador. No alteramos el archivo de Kaggle ni necesitamos descargar de nuevo un detector antiguo. Este ajuste es válido para **el clasificador de SpeciesNet utilizado después de MegaDetector V6**, no para ejecutar todos los componentes de SpeciesNet por separado.

SpeciesNet intentará clasificar el recorte del animal. Si devuelve `blank`, significa **especie no identificada**, no «tu perro está ausente».

In [ ]:
from speciesnet.classifier import SpeciesNetClassifier
import kagglehub
import json
from pathlib import Path
from shutil import copy2

animals=sorted((x for x in detections if x["class_name"]=="animal"),
               key=lambda x:x["confidence"],reverse=True)
has_people=any(x["class_name"]=="person" for x in detections)
species_suggestions=[]
species_status="not_run"
species_note=""

def prepare_classifier_model():
    """SpeciesNet 4.0.3a classifier only, using a writable metadata folder.

    KaggleHub can expose model files under read-only /kaggle/input.
    SpeciesNet's ModelInfo normally tries to put a second detector weight
    into that directory. Since our test already ran MegaDetector V6,
    this classifier-only copy does not need that detector download.
    """
    official_id="google/speciesnet/pyTorch/v4.0.3a/1"
    original=Path(kagglehub.model_download(official_id)).resolve()
    metadata=json.loads((original/"info.json").read_text(encoding="utf-8"))
    target=folder/"speciesnet_classifier_only"
    target.mkdir(parents=True,exist_ok=True)
    for key in ("classifier","classifier_labels"):
        if key not in metadata:
            raise RuntimeError("El modelo oficial no contiene "+key)
        relative=Path(metadata[key])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Ruta de pesos inesperada: "+str(relative))
        source_file=original/relative
        if not source_file.is_file():
            raise FileNotFoundError(str(source_file))
        target_file=target/relative
        target_file.parent.mkdir(parents=True,exist_ok=True)
        if not target_file.exists():
            try:
                target_file.symlink_to(source_file)
            except OSError:
                copy2(source_file,target_file)
    # Classifier-only: prevent a redundant MegaDetector V5 download into
    # read-only KaggleHub storage. We do not change original model files.
    metadata["detector"]="unused_in_classifier_only_test.pt"
    (target/"info.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
    return target

if has_people:
    species_status="person_detected"
    species_note="Hay personas en la escena. Clasificación omitida por privacidad."
elif not animals:
    species_status="no_animal"
    species_note="MegaDetector no detectó un animal en este intento."
elif animals[0]["box_fraction_of_image"]<0.01:
    species_status="insufficient_bbox"
    species_note="La caja principal es demasiado pequeña para evaluar la especie de manera fiable en esta prueba."
else:
    x1,y1,x2,y2=animals[0]["bbox_xyxy"]
    crop=photo.crop((max(0,x1),max(0,y1),min(photo.width,x2),min(photo.height,y2)))
    crop_preview=crop.copy()
    crop_preview.thumbnail((450,450))
    display(crop_preview)
    try:
        print("Preparando SpeciesNet en una carpeta temporal con permisos de escritura…")
        local_model_dir=prepare_classifier_model()
        classifier=SpeciesNetClassifier(model_name=str(local_model_dir),device=device)
        result=classifier.predict("animal.jpg",classifier.preprocess(crop))
        classes=result.get("classifications",{})
        species_suggestions=[
          {
            "label":str(label).split(";")[-1].strip(),
            "raw_label":str(label),
            "score":round(float(score),4)
          }
          for label,score in zip(classes.get("classes",[]),classes.get("scores",[]))
        ][:5]
        if not species_suggestions:
            species_status="no_prediction"
            species_note="El clasificador no entregó sugerencias."
        elif species_suggestions[0]["label"].lower()=="blank":
            species_status="not_identified"
            species_note="SpeciesNet considera que el recorte no contiene una especie identificable."
        else:
            species_status="suggested"
            species_note="Clasificación preliminar: requiere validación humana."
    except Exception as exc:
        species_status="error"
        species_note=type(exc).__name__+": "+str(exc)[:230]

print("SpeciesNet:",species_status)
print(species_note)
for suggestion in species_suggestions:
    print("  ",suggestion["label"],"—",round(suggestion["score"]*100,2),"%",
          "(estimación sin verificar)")
if species_status in ["insufficient_bbox","not_identified","no_animal"]:
    print("⚠ La especie no quedó identificada; no registrar como identificación científica.")

## 5. Resultado científico preliminar
Descarga un JSON y la fotografía marcada. No se sincroniza con PantheraLIVE, no hay diagnóstico de caza furtiva ni identificación individual.

In [ ]:
from datetime import datetime,timezone
import json
summary={
 "platform":"PantheraLAB",
 "test":"Colab MegaDetector V6 improved evaluation + SpeciesNet",
 "timestamp":datetime.now(timezone.utc).isoformat(),
 "simulation":False,
 "scientifically_verified":False,
 "source":"authorized_image_uploaded_to_colab",
 "megadetector":{"model":variant,"threshold":threshold,"detections":detections},
 "speciesnet":{"status":species_status,"note":species_note,"suggestions":species_suggestions},
 "saved_in_pantheralive":False
}
report_file=folder/"resultado_pantheralab.json"
report_file.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(summary,ensure_ascii=False,indent=2))
print("Los resultados se guardan únicamente en el entorno temporal de Colab.")

In [ ]:
from google.colab import files
files.download(str(report_file))
files.download(str(folder/"detecciones.jpg"))

**Lo siguiente para PantheraLIVE:** alojar modelos en un proceso de inferencia privado y persistente con recursos adecuados y conectar la API existente. Colab gratuito no mantiene un servicio 24/7 ni garantiza GPU. Nunca publiques un endpoint abierto que procese imágenes sensibles.